In [ ]:
"""
run_hf_bird_model.py

Batch‑process XMP side‑cars using the Hugging Face model
`chriamue/bird-species-classifier` (PyTorch).

Features:
- Loads the model locally (no external server needed).
- For each JPG, predicts the most likely bird species and a confidence.
- Low confidence (< CONF_THRESHOLD) gets a leading underscore.
- Images with no bird (confidence below a tiny threshold) are marked as "_nb".
- Adds a generic "bird" keyword to every detected‑bird image.
- Writes a CSV summary (filename, label, confidence, note).
- Updates the XMP <dc:subject> bag (creates it if missing).

Dependencies: torch, transformers, pillow, requests (for CSV writing only).
"""


In [6]:
import csv
import os
import sys
from pathlib import Path
from typing import Tuple

import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image

In [7]:
# ------------------------------------------------------------
# Configuration – adjust if needed
# ------------------------------------------------------------
MODEL_NAME = "chriamue/bird-species-classifier"
CONF_THRESHOLD = 0.80  # confidence above which we treat the label as high‑confidence
NO_BIRD_CONF = 0.10    # below this we consider the image to contain no bird

# ------------------------------------------------------------
# Load model & processor (once)
# ------------------------------------------------------------
print("🔧 Loading model…")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)
model.eval()
# Use GPU if available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)


🔧 Loading model…


Loading weights: 100%|██████████| 508/508 [00:00<00:00, 16927.03it/s]


EfficientNetForImageClassification(
  (efficientnet): EfficientNetModel(
    (embeddings): EfficientNetEmbeddings(
      (padding): ZeroPad2d((0, 1, 0, 1))
      (convolution): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=valid, bias=False)
      (batchnorm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
      (activation): SiLU()
    )
    (encoder): EfficientNetEncoder(
      (blocks): ModuleList(
        (0): EfficientNetBlock(
          (depthwise_conv): EfficientNetDepthwiseLayer(
            (depthwise_conv_pad): ZeroPad2d((0, 1, 0, 1))
            (depthwise_conv): EfficientNetDepthwiseConv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=same, groups=32, bias=False)
            (depthwise_norm): BatchNorm2d(32, eps=0.001, momentum=0.99, affine=True, track_running_stats=True)
            (depthwise_act): SiLU()
          )
          (squeeze_excite): EfficientNetSqueezeExciteLayer(
            (squeeze): AdaptiveAvgPool2d(output

In [8]:
# ------------------------------------------------------------
# Helper: predict label & confidence for a given image
# ------------------------------------------------------------
def predict(image_path: Path) -> Tuple[str, float]:
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits.squeeze(0)
    probs = torch.nn.functional.softmax(logits, dim=0)
    confidence, idx = torch.max(probs, dim=0)
    label = model.config.id2label[int(idx)]
    return label, confidence.item()

In [9]:
# ------------------------------------------------------------
# XMP handling – same as before
# ------------------------------------------------------------
import xml.etree.ElementTree as ET

def add_keywords_to_xmp(xmp_path: Path, keywords: list[str]):
    ns = {
        "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "dc": "http://purl.org/dc/elements/1.1/",
    }
    ET.register_namespace("rdf", ns["rdf"])
    ET.register_namespace("dc", ns["dc"])
    tree = ET.parse(xmp_path)
    root = tree.getroot()
    desc = root.find('.//rdf:Description', ns)
    if desc is None:
        # create a Description node under the RDF element
        rdf_elem = root.find('rdf:RDF', ns)
        if rdf_elem is None:
            rdf_elem = ET.SubElement(root, f"{{{ns['rdf']}}}RDF")
        desc = ET.SubElement(rdf_elem, f"{{{ns['rdf']}}}Description")
    # find or create dc:subject > rdf:Bag
    subject = desc.find('dc:subject', ns)
    if subject is None:
        subject = ET.SubElement(desc, f"{{{ns['dc']}}}subject")
        bag = ET.SubElement(subject, f"{{{ns['rdf']}}}Bag")
    else:
        bag = subject.find('rdf:Bag', ns)
        if bag is None:
            bag = ET.SubElement(subject, f"{{{ns['rdf']}}}Bag")
    # avoid duplicate entries
    existing = {li.text for li in bag.findall('rdf:li', ns) if li.text}
    for kw in keywords:
        if kw not in existing:
            li = ET.SubElement(bag, f"{{{ns['rdf']}}}li")
            li.text = kw
    tree.write(xmp_path, encoding='utf-8', xml_declaration=True)


In [10]:
# ------------------------------------------------------------
# Main batch processing
# ------------------------------------------------------------
def process_folder(jpg_root: Path, xmp_root: Path, csv_path: Path):
    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["filename", "label", "confidence", "note"])
        for xmp_path in xmp_root.rglob("*.xmp"):
            base = xmp_path.stem
            jpg_path = jpg_root / f"{base}.jpg"
            if not jpg_path.is_file():
                print(f"⚠️  JPEG missing for {xmp_path.name}")
                continue
            label, conf = predict(jpg_path)
            keywords = []
            note = ""
            # Determine if we think there is a bird
            if conf < NO_BIRD_CONF:
                # treat as no bird
                keywords.append("_nb")
                note = "no bird"
            else:
                # generic bird keyword
                keywords.append("bird")
                # build specific keyword: xyz-unknown-label
                xyz = label[0].lower()
                specific = f"{xyz}-unknown-{label}"
                if conf < CONF_THRESHOLD:
                    specific = f"_{specific}"  # low‑confidence marker
                keywords.append(specific)
                note = f"bird ({conf:.2f})"
            # Update XMP side‑car
            add_keywords_to_xmp(xmp_path, keywords)
            writer.writerow([xmp_path.name, label, f"{conf:.2f}", note])
            print(f"✅ {xmp_path.name} → {', '.join(keywords)} (conf={conf:.2f})")


In [11]:
if __name__ == "__main__":
    JPG_ROOT = Path("data/jpg")
    XMP_2025 = Path("data/Photos-2025")
    XMP_2024 = Path("data/Photos-2024")
    CSV_OUT = Path("output/bird_identification_summary.csv")

    print("🔧 Processing 2025 …")
    process_folder(JPG_ROOT, XMP_2025, CSV_OUT)
    print("🔧 Processing 2024 …")
    process_folder(JPG_ROOT, XMP_2024, CSV_OUT)
    print("\n✅ Done – CSV written to", CSV_OUT)


🔧 Processing 2025 …
✅ _Z9C8139.xmp → bird, _c-unknown-CALIFORNIA CONDOR (conf=0.21)
✅ _D5C6201.xmp → bird, w-unknown-WHITE TAILED TROPIC (conf=0.98)
✅ _Z9C7975.xmp → bird, _d-unknown-DALMATIAN PELICAN (conf=0.36)
✅ _Z9C7785.xmp → bird, _p-unknown-PUFFIN (conf=0.23)
✅ _D5C6177.xmp → bird, _i-unknown-IMPERIAL SHAQ (conf=0.68)
✅ _Z9C8267.xmp → bird, _c-unknown-CALIFORNIA CONDOR (conf=0.27)
✅ _D5C6348.xmp → bird, _t-unknown-TEAL DUCK (conf=0.26)
✅ _D5C6214.xmp → bird, _w-unknown-WHITE TAILED TROPIC (conf=0.47)
✅ _Z9C8104.xmp → bird, _c-unknown-CALIFORNIA CONDOR (conf=0.22)
✅ _D5C6200.xmp → bird, _w-unknown-WHITE TAILED TROPIC (conf=0.20)
✅ _Z9C8106.xmp → bird, _k-unknown-KING EIDER (conf=0.29)
✅ _Z9C7976.xmp → bird, _d-unknown-DALMATIAN PELICAN (conf=0.28)
✅ _Z9C7745.xmp → bird, _k-unknown-KING VULTURE (conf=0.47)
✅ _D5C6411.xmp → bird, r-unknown-ROCK DOVE (conf=0.88)
✅ _D5C6175.xmp → bird, _t-unknown-TEAL DUCK (conf=0.72)
✅ _Z9C7618.xmp → _nb (conf=0.09)
✅ _Z9C7793.xmp → bird, _a-unknown-